# Détection d'anomalies routières — Étape 3a : Modèles supervisés

**Notebook :** `03a_modeles_supervises.ipynb`

On entraîne trois réseaux de segmentation qui apprennent, à partir des masques annotés, à prédire
pour chaque pixel une **probabilité d'anomalie**. Ils sont construits pour que leur comparaison
réponde à deux questions précises :

| Modèle | Architecture | Encodeur | Question à laquelle il répond |
|---|---|---|---|
| **A. U-Net de zéro** | U-Net | ResNet34, poids aléatoires | Que peut apprendre un CNN avec seulement 60 images ? |
| **B. U-Net pré-entraîné** | U-Net | ResNet34, poids ImageNet | Le *transfer learning* aide-t-il ? (A vs B : seule différence = le pré-entraînement) |
| **C. DeepLabV3+ pré-entraîné** | DeepLabV3+ | ResNet34, poids ImageNet | L'architecture compte-t-elle ? (B vs C : seule différence = le décodeur) |

Les approches sans apprentissage supervisé des anomalies (modèle Cityscapes + incertitude,
autoencodeur) font l'objet des notebooks 03b et 03c.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → **GPU T4*** (indispensable).

| § | Section |
|---|---|
| 0 | Chargement : dataset, `roadanomaly_lib.py`, GPU |
| 1 | Les trois modèles |
| 2 | Fonction de perte : Dice + BCE |
| 3 | Métriques et choix du seuil |
| 4 | Boucle d'entraînement |
| 5 | Lancement : test rapide, puis les 5 tours |
| 6 | Résultats : tableau, courbes d'apprentissage, prédictions |

## 0. Chargement

Exécutez la cellule et sélectionnez **en une seule fois** (Ctrl + clic) :

- `RoadAnomaly.zip` (le dataset) ;
- `roadanomaly_lib.py` (produit par l'étape 2, dans `resultats_etape2.zip`) ;
- *facultatif* : `resultats_etape3.zip`, si une session précédente a été interrompue. Les
  entraînements déjà terminés ne seront pas refaits.

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()
RESULTS = BASE / "resultats_etape3"
(RESULTS / "preds").mkdir(parents=True, exist_ok=True)
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)

def trouver_dataset(dossier):
    dossier = Path(dossier)
    hits = sorted(dossier.rglob("frame_list.json")) if dossier.exists() else []
    return hits[0].parent if hits else None

if IN_COLAB:
    os.chdir(BASE)
    manquants = trouver_dataset(BASE / "data") is None or not (BASE / "roadanomaly_lib.py").exists()
    if manquants:
        from google.colab import files
        print("Sélectionnez RoadAnomaly.zip, roadanomaly_lib.py (et resultats_etape3.zip si reprise) :")
        for nom in files.upload():
            if nom.startswith("RoadAnomaly") and nom.endswith(".zip"):
                with zipfile.ZipFile(BASE / nom) as z: z.extractall(BASE / "data")
                os.remove(BASE / nom)
            elif nom.startswith("resultats_etape3") and nom.endswith(".zip"):
                with zipfile.ZipFile(BASE / nom) as z: z.extractall(RESULTS)
                os.remove(BASE / nom)
            elif nom.startswith("roadanomaly_lib") and nom.endswith(".py") and nom != "roadanomaly_lib.py":
                shutil.move(BASE / nom, BASE / "roadanomaly_lib.py")   # ex. "roadanomaly_lib (1).py"
    DATA_ROOT = trouver_dataset(BASE / "data")
else:
    DATA_ROOT = next((p for p in [Path("RoadAnomaly"), Path("../RoadAnomaly")]
                      if (p / "frame_list.json").exists()), None)

assert DATA_ROOT is not None, "Dataset introuvable : envoyez RoadAnomaly.zip"
assert (BASE / "roadanomaly_lib.py").exists() or Path("roadanomaly_lib.py").exists(), \
    "roadanomaly_lib.py manquant : il se trouve dans resultats_etape2.zip (étape 2)"
print("Dataset :", DATA_ROOT)
print("Entraînements déjà présents :", len(list((RESULTS / "preds").glob("*.npz"))))

**Bibliothèques.** `segmentation_models_pytorch` (smp) fournit U-Net et DeepLabV3+ avec des
encodeurs pré-entraînés. Elle n'est pas préinstallée sur Colab ; la cellule l'installe (≈ 30 s).

In [ ]:
import importlib
try:
    import segmentation_models_pytorch as smp
except ImportError:
    %pip install -q "segmentation-models-pytorch==0.6.0"
    importlib.invalidate_caches()
    import segmentation_models_pytorch as smp
try:
    import albumentations
    assert int(albumentations.__version__.split(".")[0]) >= 2
except (ImportError, AssertionError):
    %pip install -q "albumentations==2.0.8"
    importlib.invalidate_caches()

import copy, time, json, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt

sys.path.insert(0, str(BASE))
import roadanomaly_lib
importlib.reload(roadanomaly_lib)
from roadanomaly_lib import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("smp", smp.__version__, "| torch", torch.__version__, "| appareil :", device,
      torch.cuda.get_device_name(0) if device.type == "cuda" else "")
if device.type != "cuda":
    warnings.warn("Pas de GPU : activez-le (Exécution → Modifier le type d'exécution → GPU T4). "
                  "Sur CPU, seul le test rapide est raisonnable.")

splits = make_splits(DATA_ROOT)
print(f"{len(splits)} images, plis :", splits.fold.value_counts().sort_index().tolist())

plt.rcParams.update({"figure.dpi": 80, "savefig.dpi": 150, "savefig.bbox": "tight",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "axes.axisbelow": True})

## 1. Les trois modèles

### Structure commune : encodeur + décodeur

Un réseau de segmentation a deux parties :

- l'**encodeur** réduit progressivement l'image (1/2, 1/4, … 1/32) et en extrait des
  caractéristiques de plus en plus abstraites : contours, puis textures, puis parties d'objets ;
- le **décodeur** remonte à la résolution de l'image et produit **une valeur par pixel**
  (un *logit*, transformé en probabilité par la fonction sigmoïde).

Les trois modèles utilisent le **même encodeur, ResNet34** (21 millions de paramètres, 34
couches avec connexions résiduelles), ce qui rend la comparaison équitable.

### Pourquoi U-Net ?

U-Net ajoute des **connexions de saut** (*skip connections*) : les caractéristiques de chaque
niveau de l'encodeur sont recopiées vers le niveau correspondant du décodeur. Le décodeur
retrouve ainsi les **détails fins** perdus lors de la réduction. C'est essentiel ici : l'étape 1 a
montré que beaucoup d'objets (cônes, rochers) sont petits. U-Net a été conçu pour des
**petits datasets** d'images médicales, une situation proche de la nôtre.

### Pourquoi DeepLabV3+ ?

DeepLabV3+ utilise des **convolutions dilatées** (*atrous*) et un module **ASPP** qui analyse
l'image à plusieurs échelles en parallèle. Il capte mieux le **contexte** (la route autour de
l'objet), utile pour des objets de tailles très différentes (de 100 à 300 000 px). Son décodeur,
plus léger que celui d'U-Net, récupère les détails à partir d'un seul niveau intermédiaire
(1/4).

### Pourquoi un U-Net de zéro ?

Il sert de **référence**. Même architecture que B, mais poids aléatoires : la différence de score
entre A et B mesure exactement l'apport du pré-entraînement sur ImageNet (1,2 million d'images).

In [ ]:
ENCODER_WEIGHTS = None if os.environ.get("RA_NO_PRETRAINED") else "imagenet"   # tests hors Colab

MODELS = {
    "A_unet_scratch":  dict(arch="Unet",          pretrained=False, label="A. U-Net de zéro"),
    "B_unet_imagenet": dict(arch="Unet",          pretrained=True,  label="B. U-Net pré-entraîné"),
    "C_deeplabv3plus": dict(arch="DeepLabV3Plus", pretrained=True,  label="C. DeepLabV3+ pré-entraîné"),
}
COLORS = {"A_unet_scratch": "#8a8984", "B_unet_imagenet": "#2a78d6", "C_deeplabv3plus": "#eb6834"}

def build_model(name):
    cfg = MODELS[name]
    Arch = getattr(smp, cfg["arch"])
    return Arch(encoder_name="resnet34",
                encoder_weights=ENCODER_WEIGHTS if cfg["pretrained"] else None,
                in_channels=3, classes=1)       # 1 sortie : logit d'anomalie par pixel

for name in MODELS:
    m = build_model(name)
    n_enc = sum(p.numel() for p in m.encoder.parameters()) / 1e6
    n_tot = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"{MODELS[name]['label']:<28} {n_tot:5.1f} M paramètres (encodeur {n_enc:.1f} M, décodeur {n_tot - n_enc:.1f} M)")
del m

## 2. Fonction de perte : Dice + BCE

L'étape 1 a montré que seuls **9,85 % des pixels** sont anomaux. Avec l'entropie croisée binaire
(**BCE**) seule, chaque pixel compte autant : le modèle peut obtenir une perte faible en prédisant
« fond » presque partout. On combine donc deux pertes :

```
perte = BCE + Dice
```

- **BCE** (*binary cross-entropy*) : pénalise chaque pixel mal classé. Gradients stables, bonne
  calibration des probabilités.
- **Dice loss** = 1 − Dice, avec Dice = 2·|P ∩ V| / (|P| + |V|) calculé sur les probabilités.
  Elle mesure le **recouvrement** entre prédiction P et vérité V. Elle ne dépend pas du nombre de
  pixels de fond correctement classés : une anomalie de 200 px manquée la fait fortement monter,
  quelle que soit la taille de l'image. C'est la réponse directe au déséquilibre.

In [ ]:
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()

def criterion(logits, target):
    return bce_loss(logits, target) + dice_loss(logits, target)

## 3. Métriques et choix du seuil

Le modèle donne une probabilité par pixel ; un **seuil** la transforme en décision. Pour comparer
les seuils sans tout recalculer, on range les probabilités (arrondies à 1/255) dans deux
**histogrammes** : un pour les pixels réellement anomaux, un pour les pixels de fond. Pour
n'importe quel seuil, on en déduit directement :

| Quantité | Définition |
|---|---|
| TP (vrais positifs) | pixels anomaux prédits anomaux |
| FP (faux positifs) | pixels de fond prédits anomaux (fausses alarmes) |
| FN (faux négatifs) | pixels anomaux manqués |
| Précision | TP / (TP + FP) |
| Rappel | TP / (TP + FN) |
| F1 = Dice | 2·TP / (2·TP + FP + FN) |
| IoU | TP / (TP + FP + FN) |

Les métriques sont calculées sur **tous les pixels de l'ensemble** (validation ou test), à
960 × 540, en excluant les 4 lignes de complétion.

**Choix du seuil :** il est fixé sur la **validation** (celui qui maximise le F1), jamais sur le
test : choisir le seuil sur le test reviendrait à « tricher ».

In [ ]:
def predict(model, loader, keep=False):
    # Prédit chaque image ; renvoie les histogrammes (anomalie, fond) et, si keep, les probabilités
    model.eval()
    pos, neg, probs = np.zeros(256, np.int64), np.zeros(256, np.int64), {}
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16,
                                         enabled=device.type == "cuda"):
        for b in loader:
            p = torch.sigmoid(model(b["image"].to(device)).float())[0, 0, :WORK_H, :WORK_W]
            p8 = (p * 255).round().to(torch.uint8).cpu().numpy()
            m = b["mask"][0, 0, :WORK_H, :WORK_W].numpy().astype(bool)
            pos += np.bincount(p8[m], minlength=256)
            neg += np.bincount(p8[~m], minlength=256)
            if keep:
                probs[b["name"][0]] = p8
    return pos, neg, probs

def metrics_at(pos, neg, t):
    # métriques au seuil t (entier 0..255 : pixel anomal si probabilité*255 >= t)
    tp, fn, fp = pos[t:].sum(), pos[:t].sum(), neg[t:].sum()
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    return dict(precision=prec, recall=rec, f1=f1, iou=iou)

def best_threshold(pos, neg):
    # seuil qui maximise le F1 (calcul vectorisé sur les 255 seuils possibles)
    tp = pos[::-1].cumsum()[::-1]; fp = neg[::-1].cumsum()[::-1]; fn = pos.sum() - tp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    t = int(np.argmax(f1[1:]) + 1)
    return t, f1[t]

# petit test sur des histogrammes fictifs
_pos, _neg = np.bincount([200, 220, 240, 100], minlength=256), np.bincount([10, 20, 150], minlength=256)
print(metrics_at(_pos, _neg, 128), "| meilleur seuil :", best_threshold(_pos, _neg))

## 4. Boucle d'entraînement

| Réglage | Valeur | Pourquoi |
|---|---|---|
| Optimiseur | AdamW, *weight decay* 1e-4 | standard, robuste ; le *weight decay* limite le sur-apprentissage |
| Taux d'apprentissage | décodeur 1e-3 ; encodeur 1e-3 (A) ou 1e-4 (B, C) | un encodeur pré-entraîné doit être ajusté doucement, pour ne pas effacer ce qu'il sait |
| Encodeur gelé | 3 premières époques (B, C) | le décodeur, initialisé au hasard, produit d'abord des gradients désordonnés qui abîmeraient l'encodeur |
| Planning | décroissance cosinus sur 40 époques | grands pas au début, ajustements fins à la fin |
| Lots | 4 découpes 512 × 512, 4 découpes par image et par époque | mémoire du GPU T4 ; variété |
| Précision mixte | float16 sur GPU | environ 2 fois plus rapide, même qualité |
| Arrêt anticipé | meilleure IoU de validation ; arrêt après 12 époques sans progrès | on garde le modèle qui généralise le mieux, pas le dernier |

À la fin de chaque entraînement : seuil choisi sur la validation, puis prédiction du **test**.
Les probabilités du test sont sauvegardées (`preds/<modèle>_fold<k>.npz`) : l'étape 4 calculera
toutes les métriques à partir de ces fichiers, sans réentraîner.

In [ ]:
class Config:
    epochs = 40
    freeze_epochs = 3
    patience = 12
    batch_size = 4
    samples_per_image = 4
    lr_decoder = 1e-3
    lr_encoder_pretrained = 1e-4
    lr_encoder_scratch = 1e-3

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_one(name, k, cfg, verbose=True):
    set_seed(SEED + k)
    pretrained = MODELS[name]["pretrained"]
    train_loader, val_loader, test_loader = make_loaders(
        DATA_ROOT, splits, k, batch_size=cfg.batch_size, samples_per_image=cfg.samples_per_image,
        num_workers=2)
    model = build_model(name).to(device)
    enc = list(model.encoder.parameters())
    dec = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    lr_enc = cfg.lr_encoder_pretrained if pretrained else cfg.lr_encoder_scratch
    opt = torch.optim.AdamW([{"params": enc, "lr": lr_enc}, {"params": dec, "lr": cfg.lr_decoder}],
                            weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, best_iou, best_state, best_epoch, t0 = [], -1.0, None, 0, time.time()
    for epoch in range(1, cfg.epochs + 1):
        frozen = pretrained and epoch <= cfg.freeze_epochs
        for p in enc:
            p.requires_grad = not frozen
        model.train()
        if frozen:
            model.encoder.eval()        # garde aussi les statistiques BatchNorm d'ImageNet
        total, count = 0.0, 0
        for b in train_loader:
            x, y = b["image"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item() * len(x); count += len(x)
        sched.step()

        pos, neg, _ = predict(model, val_loader)
        val = metrics_at(pos, neg, 128)                 # seuil 0,5 pendant l'entraînement
        history.append(dict(model=name, fold=k, epoch=epoch, train_loss=total / count,
                            val_iou=val["iou"], val_f1=val["f1"], frozen=frozen))
        if val["iou"] > best_iou:
            best_iou, best_epoch = val["iou"], epoch
            best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
        if verbose:
            print(f"  époque {epoch:2d}  perte {total / count:.3f}  IoU val {val['iou']:.3f}"
                  f"{'  (encodeur gelé)' if frozen else ''}{'  *' if best_epoch == epoch else ''}")
        if epoch - best_epoch >= cfg.patience:
            if verbose: print(f"  arrêt anticipé : pas de progrès depuis l'époque {best_epoch}")
            break

    model.load_state_dict(best_state)
    pos, neg, _ = predict(model, val_loader)
    t, _ = best_threshold(pos, neg)
    val_best = metrics_at(pos, neg, t)
    pos_t, neg_t, probs = predict(model, test_loader, keep=True)
    test = metrics_at(pos_t, neg_t, t)

    np.savez_compressed(RESULTS / "preds" / f"{name}_fold{k}.npz", **probs)
    summary = dict(model=name, fold=k, best_epoch=best_epoch, epochs_run=len(history),
                   threshold=t / 255, val_iou=val_best["iou"], val_f1=val_best["f1"],
                   test_iou=test["iou"], test_f1=test["f1"], test_precision=test["precision"],
                   test_recall=test["recall"], minutes=(time.time() - t0) / 60)
    pd.DataFrame(history).to_csv(RESULTS / f"history_{name}_fold{k}.csv", index=False)
    json.dump(summary, open(RESULTS / f"summary_{name}_fold{k}.json", "w"), indent=1)
    del model, opt; torch.cuda.empty_cache()
    return summary

## 5. Lancement

### 5.1 Test rapide (≈ 2 minutes sur GPU)

Avant de lancer 15 entraînements, on vérifie que tout fonctionne : 2 époques par modèle, tour 0
uniquement. Les scores n'ont **aucune valeur** ici ; on vérifie seulement l'absence d'erreur.
Les fichiers de ce test sont ensuite supprimés.

In [ ]:
class Quick(Config):
    epochs, freeze_epochs, samples_per_image = 2, 1, 1

t0 = time.time()
for name in MODELS:
    print(MODELS[name]["label"])
    s = train_one(name, 0, Quick)
    print(f"  -> OK, seuil {s['threshold']:.2f}, IoU test {s['test_iou']:.3f}\n")
print(f"Test rapide terminé en {(time.time() - t0) / 60:.1f} min")
for f in list(RESULTS.glob("*fold0*")) + list((RESULTS / "preds").glob("*fold0*")):
    f.unlink()

### 5.2 Entraînement complet : 3 modèles × 5 tours

Durée estimée sur un GPU T4 : **3 à 6 minutes par entraînement, soit environ 1 heure** pour les
15. Chaque entraînement terminé est sauvegardé ; si la session est interrompue, relancez le
notebook en envoyant aussi `resultats_etape3.zip` (téléchargé à la fin, ou par la cellule de
sauvegarde ci-dessous) : les entraînements déjà faits sont sautés.

Pour aller plus vite pendant la mise au point, vous pouvez réduire `FOLDS = [0]`.

In [ ]:
FOLDS = [0, 1, 2, 3, 4]

for k in FOLDS:
    for name in MODELS:
        out = RESULTS / f"summary_{name}_fold{k}.json"
        if out.exists():
            print(f"[déjà fait] {MODELS[name]['label']} — tour {k}")
            continue
        print(f"=== {MODELS[name]['label']} — tour {k} ===")
        s = train_one(name, k, Config)
        print(f"-> meilleure époque {s['best_epoch']}, seuil {s['threshold']:.2f} | "
              f"IoU val {s['val_iou']:.3f} | IoU test {s['test_iou']:.3f} | {s['minutes']:.1f} min\n")

In [ ]:
# Sauvegarde intermédiaire : à exécuter à tout moment pour récupérer ce qui est déjà fait
shutil.make_archive(str(BASE / "resultats_etape3"), "zip", RESULTS)
print("Archive :", BASE / "resultats_etape3.zip")
if IN_COLAB:
    from google.colab import files
    files.download(str(BASE / "resultats_etape3.zip"))

## 6. Résultats

### 6.1 Scores par modèle et par tour

Ces scores sont calculés sur les pixels de test de chaque tour, au seuil choisi sur la
validation. L'évaluation complète (AUROC, AUPRC, résultats par catégorie et par taille d'objet)
est faite à l'étape 4.

In [ ]:
summ = pd.DataFrame([json.load(open(f)) for f in sorted(RESULTS.glob("summary_*.json"))])
summ["modèle"] = summ.model.map(lambda m: MODELS[m]["label"])

par_tour = summ.pivot_table(index="modèle", columns="fold", values="test_iou").round(3)
par_tour.columns = [f"tour {c}" for c in par_tour.columns]
display(par_tour)

agg = summ.groupby("modèle")[["test_iou", "test_f1", "test_precision", "test_recall", "threshold",
                              "best_epoch", "minutes"]].agg(["mean", "std"])
pm = lambda m, s: f"{m:.3f} ± {s:.3f}" if s == s else f"{m:.3f} ± —"     # s != s : un seul tour
tab = pd.DataFrame(index=agg.index, data={
    "IoU test": [pm(m, s) for m, s in zip(agg.test_iou["mean"], agg.test_iou["std"])],
    "F1 (= Dice) test": [pm(m, s) for m, s in zip(agg.test_f1["mean"], agg.test_f1["std"])],
    "précision": agg.test_precision["mean"].map("{:.3f}".format),
    "rappel": agg.test_recall["mean"].map("{:.3f}".format),
    "seuil moyen": agg.threshold["mean"].map("{:.2f}".format),
    "meilleure époque": agg.best_epoch["mean"].map("{:.0f}".format),
    "minutes / tour": agg.minutes["mean"].map("{:.1f}".format),
})
display(tab)
summ.to_csv(RESULTS / "resume_modeles_supervises.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
names = [n for n in MODELS if n in set(summ.model)]
for i, name in enumerate(names):
    v = summ.loc[summ.model == name, "test_iou"].values
    ax.scatter(v, np.full(len(v), i), s=50, color=COLORS[name], edgecolor="white", zorder=3)
    ax.plot([v.mean()] * 2, [i - 0.3, i + 0.3], color="#2b2b2b", lw=2, zorder=4)
    ax.text(v.mean(), i + 0.38, f"moyenne {v.mean():.3f}", ha="center", fontsize=9)
ax.set_yticks(range(len(names))); ax.set_yticklabels([MODELS[n]["label"] for n in names])
ax.invert_yaxis(); ax.set_xlabel("IoU sur le test (un point = un tour)"); ax.grid(axis="y", visible=False)
ax.set_title("IoU de test des modèles supervisés sur les 5 tours", fontweight="bold")
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "01_iou_par_modele.png"); plt.show()

### 6.2 Courbes d'apprentissage (tour 0)

À gauche, la perte d'entraînement ; à droite, l'IoU de validation. Un écart qui se creuse
(perte qui baisse, IoU de validation qui stagne ou baisse) est le signe du **sur-apprentissage**.
La zone grisée marque les époques où l'encodeur est gelé (B et C).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.8))
for name in MODELS:
    f = RESULTS / f"history_{name}_fold0.csv"
    if not f.exists():
        continue
    h = pd.read_csv(f)
    a1.plot(h.epoch, h.train_loss, color=COLORS[name], lw=2, label=MODELS[name]["label"])
    a2.plot(h.epoch, h.val_iou, color=COLORS[name], lw=2, label=MODELS[name]["label"])
    best = h.loc[h.val_iou.idxmax()]
    a2.scatter(best.epoch, best.val_iou, color=COLORS[name], s=50, zorder=3, edgecolor="white")
for a in (a1, a2):
    a.axvspan(0.5, Config.freeze_epochs + 0.5, color="#e4e3df", alpha=0.6, lw=0)
    a.set_xlabel("Époque"); a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
a1.set_ylabel("Perte (BCE + Dice)"); a1.set_title("Perte d'entraînement", fontweight="bold")
a2.set_ylabel("IoU (seuil 0,5)"); a2.set_title("IoU de validation (point = meilleure époque)", fontweight="bold")
a2.legend(frameon=False, fontsize=9)
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "02_courbes_apprentissage_tour0.png"); plt.show()

### 6.3 Exemples de prédictions (images de test du tour 0)

Pour chaque image : l'image, la vérité terrain, puis la **carte de probabilité** de chaque modèle
(blanc = probabilité d'anomalie élevée). Le contour cyan est la vérité terrain.

In [ ]:
import cv2
preds = {n: np.load(RESULTS / "preds" / f"{n}_fold0.npz") for n in MODELS
         if (RESULTS / "preds" / f"{n}_fold0.npz").exists()}
test0 = fold_sets(splits, 0)[2]
choix = test0[:: max(1, len(test0) // 4)][:4]       # 4 images réparties dans le pli de test

fig, axes = plt.subplots(len(choix), 2 + len(preds), figsize=(3.4 * (2 + len(preds)), 2.1 * len(choix)))
axes = np.atleast_2d(axes)
for r, n in enumerate(choix):
    img = cv2.resize(load_image(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_AREA)
    msk = cv2.resize(load_mask(DATA_ROOT, n), (WORK_W, WORK_H), interpolation=cv2.INTER_NEAREST)
    cont, _ = cv2.findContours(msk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    axes[r, 0].imshow(img); axes[r, 1].imshow(msk, cmap="gray")
    axes[r, 0].set_ylabel(n[:20], fontsize=8)
    for c, (name, P) in enumerate(preds.items(), start=2):
        vis = cv2.cvtColor(P[n], cv2.COLOR_GRAY2RGB)
        axes[r, c].imshow(cv2.drawContours(vis, cont, -1, (0, 200, 255), 2))
        if r == 0: axes[r, c].set_title(MODELS[name]["label"], fontsize=9)
    if r == 0:
        axes[r, 0].set_title("Image", fontsize=9); axes[r, 1].set_title("Vérité terrain", fontsize=9)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); fig.savefig(RESULTS / "figures" / "03_exemples_predictions_tour0.png"); plt.show()

### 6.4 Lecture des résultats

Interprétez vos chiffres avec ces questions (les réponses serviront au rapport) :

1. **A vs B — l'apport du transfer learning.** L'écart d'IoU entre le U-Net de zéro et le U-Net
   pré-entraîné est généralement **très important** avec si peu d'images. Sur les courbes, le
   modèle A apprend plus lentement et son IoU de validation reste basse : il doit tout apprendre
   (contours, textures, formes) à partir de 35 images.
2. **B vs C — l'architecture.** L'écart est en général bien plus faible qu'entre A et B. U-Net
   peut avoir l'avantage sur les petits objets (connexions de saut à tous les niveaux),
   DeepLabV3+ sur les grands (contexte multi-échelle). L'étape 4 le vérifiera par taille d'objet.
3. **Variabilité entre tours.** Un écart-type élevé signifie que le score dépend beaucoup des
   images testées : c'est attendu avec ≈ 12 images par pli, et c'est pourquoi on fait 5 tours.
   Le tour 3 (pneus jamais vus à l'entraînement) est souvent le plus difficile.
4. **Précision vs rappel.** Un rappel faible signifie des anomalies manquées (dangereux pour un
   véhicule) ; une précision faible, des fausses alarmes.

**Étape suivante :** `03b` (modèle entraîné sur Cityscapes + score d'incertitude) et `03c`
(autoencodeur), puis l'évaluation complète à l'étape 4.

In [ ]:
# Archive finale : gardez resultats_etape3.zip pour l'étape 4
shutil.make_archive(str(BASE / "resultats_etape3"), "zip", RESULTS)
if IN_COLAB:
    from google.colab import files
    files.download(str(BASE / "resultats_etape3.zip"))
print(sorted(str(p.relative_to(RESULTS)) for p in RESULTS.rglob("*") if p.is_file()))